In [2]:
import openai
import json
import random
from pydantic import BaseModel, Field
from typing import List


In [12]:
api_key = ""
client = openai.OpenAI(api_key=api_key) # обошлось в 30 центов

In [13]:
class QuestionList(BaseModel):
    questions: List[str] = Field(description="1-2 конкретных технических вопросов по текст")

def generate_questions(chunk_content):
    system_prompt = """Вы — эксперт по тестированию поисковых систем (QA). 
    Ваша задача: прочитать фрагмент технической документации и составить на его основе 1-2 вопроса.

    Требования к вопросам:
    1. Вопрос должен быть таким, чтобы ответом на него был ИМЕННО этот фрагмент текста.
    2. Используйте конкретные названия моделей (HW1000), команд (iplir...) или цифры из таблиц.
    3. НИКАКИХ ответов присылать не нужно. Только вопросы.
    4. Избегайте общих вопросов типа "О чем этот раздел?"."""

    try:
        response = client.beta.chat.completions.parse(
            model="gpt-4.1",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Создай вопросы к тексту:\n\n{chunk_content}"}
            ],
            response_format=QuestionList,
            temperature=0.5
        )
        
        return response.choices[0].message.parsed.questions
    except Exception as e:
        print(f"Ошибка: {e}")
        return []



In [14]:
file_path = "exported_chunks.json"
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)
        
print(f"Успешно загружено {len(data)} чанков из файла {file_path}")
    
test_chunks = random.sample(data, 200)
test_chunks[0]

Успешно загружено 3450 чанков из файла exported_chunks.json


{'id': 'id_1646',
 'content': 'Раздел: Режимы командного интерпретатора\nРежим настройки.\n\nРаздел: Особенности использования\nЗначение административной дистанции для маршрутов по умолчанию можно задать только вместе с административной дистанцией для всего протокола DHCP.\n\nРаздел: Пример использования\nhostname# inet dhcp client route-distance 80 default-route 60\nSet distance to 80, default distance to 60',
 'metadata': {'type': 'text', 'page': 128, 'context': ''}}

In [15]:
benchmark_tasks = []
for chunk in test_chunks:
    qs = generate_questions(chunk['content'])
    print(f"{qs}\n\n")
    for q in qs:
        benchmark_tasks.append({
            "question": q,
            "expected_page": chunk["metadata"].get("page"),
            "expected_context": chunk["metadata"].get("context"),
            "original_content": chunk["content"]
        })

['Как можно задать значение административной дистанции для маршрутов по умолчанию в протоколе DHCP?', 'Какая команда используется для установки дистанции 80 и дистанции по умолчанию 60 для клиента DHCP?']


['Какая аппаратная платформа была добавлена в поддержку в версии 5.3.0?', 'Какой USB-токен теперь поддерживается для аутентификации в версии 5.3.0?']


['При каких условиях IP-пакет будет перенаправлен на ViPNet Coordinator HW и обработан как локальный трафик открытой сети при работе прокси-сервера в «прозрачном» режиме?', 'Какие прикладные протоколы (DNS, FTP, H.323, SCCP, SIP) должны быть включены для применения трансляции к соответствующим IP-пакетам согласно разделу о принципах фильтрации трафика?']


['Какой идентификатор OID у VIPNETHW-MIB, предназначенного для получения информации о службе remote_management?', 'Какая ветвь используется для получения информации о службе внешнего управления remote_management в VIPNETHW-MIB?']


['Что происходит с остальными параметрами в секции

In [16]:
benchmark_tasks

[{'question': 'Как можно задать значение административной дистанции для маршрутов по умолчанию в протоколе DHCP?',
  'expected_page': 128,
  'expected_context': '',
  'original_content': 'Раздел: Режимы командного интерпретатора\nРежим настройки.\n\nРаздел: Особенности использования\nЗначение административной дистанции для маршрутов по умолчанию можно задать только вместе с административной дистанцией для всего протокола DHCP.\n\nРаздел: Пример использования\nhostname# inet dhcp client route-distance 80 default-route 60\nSet distance to 80, default distance to 60'},
 {'question': 'Какая команда используется для установки дистанции 80 и дистанции по умолчанию 60 для клиента DHCP?',
  'expected_page': 128,
  'expected_context': '',
  'original_content': 'Раздел: Режимы командного интерпретатора\nРежим настройки.\n\nРаздел: Особенности использования\nЗначение административной дистанции для маршрутов по умолчанию можно задать только вместе с административной дистанцией для всего протокола 

In [17]:
with open("benchmark_tasks.json", "w", encoding="utf-8") as f:
        json.dump(benchmark_tasks, f, ensure_ascii=False, indent=2)